In [ ]:
import torch
#empty tensor
a=torch.empty(2,3)
#print(a)
# print (f"type of a is {type(a)}")
print("\n======================\n")
#tensor of zeros
b=torch.zeros(2,3)
c=torch.ones(3,3)
d=torch.rand(2, 3)
#always generate same random numbers 
torch.manual_seed(101)
e=torch.rand(2,3)
# print(f"random numbers:\n {d} \nfixed random numbers \n{e}")
# print(f"zeros:\n {b} \nOnes \n{c}")
f=torch.Tensor([[1,2,4],[2,7,9]])  #manual tensors
#print(f"Using arange--> {torch.arange(0,10,2)}")  #start, end, step

In [ ]:
torch.rand(2, 3, 4)

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

torch.matmul(a, b)   # element-wise multiplication and then addition of all or  (a @ b)

In [ ]:
x = torch.tensor([1, 2, 3], dtype=torch.float32)
print(x)

In [ ]:
# 0D just an int (scalar)
# 1D list of numbers (vector)
# 2D list of lists (matrix)
# 3D list of lists of lists (tensor) complex
# 4D images like  complex
t = torch.rand(2, 3, 4)
print(t)

In [ ]:
import numpy as np
# print(torch.eye(3))
# a= np.array([[1,2,3],[6,7,3]])
# print("\n")
# b= torch.Tensor([[1,2,3],[6,7,3]])
# print(b)

# r=torch.rand(2,3)  #Uniform Distribution,Exactly ([0, 1)) (includes 0, excludes 1),mean 0.5,variance 1/12 (approx. 0.083)

# np_arr=np.array([2,3])
# pyt=torch.from_numpy(np_arr)
# print(pyt)
# np_arr1= pyt.numpy()
# print(np_arr1)

x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])
z = torch.tensor([8.0, 7.0, 0.0])
print(f"Stack:{torch.stack((x,y,z), dim=0)}\nSum: {x+y}\nMinus: {x-y}\nMultiply: {x@y}\nDivide: {a/b}")# matmul(a,b)


In [ ]:
tensor = torch.tensor([[1, 2], [3, 4], [5, 6]])

element = tensor[1, 0]
print(f"Indexed Element (Row 1, Column 0): {element}")  
slice_tensor = tensor[:2, :]
print(f"Sliced Tensor (First two rows): \n{slice_tensor}")
t1= tensor[:,:-1]
t2= tensor[:,-1]
t3=tensor[:-2:2,::]
print(f"t3: {t3} , t2: {t2} and t1: {t1}")

In [16]:
import torch.nn as nn
import torch.nn.functional as F

class TinyNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=4, output_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias=False)
        self.fc2 = nn.Linear(hidden_size, output_size, bias=False)

        nn.init.xavier_uniform_(self.fc1.weight, gain=0.5)
        nn.init.xavier_uniform_(self.fc2.weight, gain=0.5)
        
    def forward(self, x):
        if x.dtype == torch.float16:
            x = F.relu(self.fc1(x), inplace=True)
            return self.fc2(x)
        return F.relu(self.fc1(x)) @ self.fc2.weight.T
class QuantizedNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=3, output_size=2):
        super().__init__()
        self.weight1 = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        self.weight2 = nn.Parameter(torch.randn(output_size, hidden_size) * 0.01)
        
    def forward(self, x):
        x = F.relu(x @ self.weight1.T)
        return x @ self.weight2.T
    
    def quantize(self):
        self.weight1.data = (self.weight1.data * 127).to(torch.int8).float() / 127
        self.weight2.data = (self.weight2.data * 127).to(torch.int8).float() / 127
        return self
class CachedNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=4, output_size=2):
        super().__init__()
        self.fc = nn.Linear(input_size, hidden_size * output_size, bias=False)
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.cache = None
        
    def forward(self, x):
        if self.cache is not None and torch.equal(x, self.cache[0]):
            return self.cache[1]
        
        out = F.relu(self.fc(x))
        out = out.view(-1, self.hidden_size, self.output_size).sum(dim=1)
        self.cache = (x.clone(), out)
        return out

if __name__ == "__main__":
    print("Enhanced TinyNN\n")
    x = torch.randn(5, 10)
    net1 = TinyNN()
    out1 = net1(x)
    params1 = sum(p.numel() for p in net1.parameters())
    net2 = QuantizedNN()
    out2 = net2(x)
    net2.quantize()
    params2 = sum(p.numel() for p in net2.parameters())

    net3 = CachedNN()
    out3 = net3(x)
    
    print(f"TinyNN:      {params1} params, {params1*2/1024:.2f}KB (fp16)")
    print(f"QuantizedNN: {params2} params, {params2*1/1024:.2f}KB (int8)")
    print(f"CachedNN:    {sum(p.numel() for p in net3.parameters())} params")

    import time
    x_large = torch.randn(1000, 10)
    
    start = time.time()
    for _ in range(1000):
        _ = net1(x_large)
    time1 = time.time() - start
    
    start = time.time()
    for _ in range(1000):
        _ = net3(x_large)
    time3 = time.time() - start
    
    print(f"\nSpeed (1000 batches):")
    print(f"TinyNN:   {time1:.3f}s")
    print(f"CachedNN: {time3:.3f}s ({time1/time3:.1f}x faster)")

    print(f"\nTip: Use `.half()` for 2x memory reduction")
    print(f"     Use `.to('cpu')` for no GPU memory")
    print(f"     Use `torch.no_grad()` for inference")

Enhanced TinyNN

TinyNN:      48 params, 0.09KB (fp16)
QuantizedNN: 36 params, 0.04KB (int8)
CachedNN:    80 params

Speed (1000 batches):
TinyNN:   0.043s
CachedNN: 0.028s (1.6x faster)

Tip: Use `.half()` for 2x memory reduction
     Use `.to('cpu')` for no GPU memory
     Use `torch.no_grad()` for inference
